# PubMed — API & bulk

PubMed is NLM's 40 M+ biomedical citation database. `biodb.pubmed` wraps
both modes:

| Mode | Functions | Use case |
|---|---|---|
| **NCBI E-utilities** | `search`, `query_pmid`, `query_summaries`, `query_abstract` | One paper / a few papers — search, summary metadata, full abstract |
| **Bulk XML.gz** | `list_baseline_files`, `download_baseline_file`, `list_update_files`, `download_update_file`, `parse_pubmed_xml` | Whole-corpus mining — annual baseline + daily updates |

The API path is rate-limit-friendly out of the box (340 ms per call →
stays under NCBI's un-keyed 3 req/sec cap). Pass `api_key=` to lift to
10 req/sec.

In [1]:
from biodb import pubmed

## 1. API mode — search → summary → full abstract

`search` runs an `esearch` against `db=pubmed` and returns PMIDs plus
the **full match count** (even when the page is truncated). NCBI's
query translator expands free-text terms into MeSH-aware boolean
queries automatically — handy for users who don't write
`"breast neoplasms"[MeSH Terms] AND BRCA1[gene]` by hand.

In [2]:
hits = pubmed.search("BRCA1", retmax=5)
print(f"matched {hits['total']:,} citations; first 5 PMIDs: {hits['pmids']}")
print(f"query translation: {hits['query_translation'][:200]}…")

matched 25,507 citations; first 5 PMIDs: ['42147998', '42147479', '42147175', '42146504', '42145025']
query translation: "brca1 protein human"[Supplementary Concept] OR "brca1 protein human"[All Fields] OR "brca1"[All Fields] OR "genes, brca1"[MeSH Terms] OR ("genes"[All Fields] AND "brca1"[All Fields]) OR "brca1 genes"…


`query_summaries` batches the `esummary` lookup for a list of PMIDs.
Columns: `pmid`, `title`, `authors`, `source` (ISO journal abbreviation),
`pubdate`, `epubdate`, `doi`.

In [3]:
summaries = pubmed.query_summaries(hits["pmids"])
summaries[["pmid", "source", "pubdate", "title"]]

,pmid,source,pubdate,title
0,42147998,Balkan J Med Genet,2025 Dec,Secretory Breast Carcinoma in A Six-Year-Old G...
1,42147479,Explor Target Antitumor Ther,2026,Triple-negative breast cancer: current underst...
2,42147175,Res Sq,2026 May 5,Familial Disclosure and Cascade Testing in Hig...
3,42146504,bioRxiv,2026 May 10,USP1 inhibition promotes RAD18-dependent PCNA ...
4,42145025,Acta Oncol,2026 May 18,Exploring the somatic mutational landscape of ...


`query_abstract` does an `efetch` and parses the full PubMed XML into
a single record — title, abstract, authors, journal, pub year, DOI,
MeSH terms.

In [4]:
record = pubmed.query_abstract("30700171")
{
    "pmid": record["pmid"],
    "title": record["title"][:90] + "…",
    "journal": record["journal"],
    "pub_year": record["pub_year"],
    "doi": record["doi"],
    "authors[:3]": record["authors"][:3],
    "mesh_terms[:5]": record["mesh_terms"][:5],
    "abstract_length": len(record["abstract"]),
}

{'pmid': '30700171',
 'title': '\'Relationship between thermal dose and cell death for "rapid" ablative and "slow" hyperthe…',
 'journal': 'Int J Hyperthermia',
 'pub_year': '2019',
 'doi': '10.1080/02656736.2018.1558289',
 'authors[:3]': ['Mouratidis PXE', 'Rivens I', 'Civale J'],
 'mesh_terms[:5]': ['Cell Death',
  'Heating',
  'Humans',
  'Hyperthermia, Induced'],
 'abstract_length': 1778}

Labeled abstract sections (`AIM`, `METHODS`, `RESULTS`, `CONCLUSIONS`)
are concatenated with their labels so the structure isn't lost.

In [5]:
print(record["abstract"][:500] + "…")

AIM: Thermal isoeffective dose (TID) has not been convincingly validated for application to predict biological effects from rapid thermal ablation (e.g., using >55 °C). This study compares the classical method of quantifying TID (derived from hyperthermia data) with a temperature-adjusted method based on the Arrhenius model for predicting cell survival in vitro, after either 'rapid' ablative or 'slow' hyperthermic exposures.

METHODS: MTT assay viability data was obtained from two human colon ca…


## 2. Bulk mode — Annual Baseline + Daily Updates

Each December, NLM releases the full PubMed corpus as ~1,300
`pubmed{YY}n####.xml.gz` shards (~20 MB each, ~30 GB total compressed)
at <https://ftp.ncbi.nlm.nih.gov/pubmed/baseline/>. Daily update files
at <https://ftp.ncbi.nlm.nih.gov/pubmed/updatefiles/> continue
numerically from the baseline with new / revised / deleted citations.

`list_baseline_files` scrapes the directory listing — handy for
distributing shards across workers.

In [6]:
baseline = pubmed.list_baseline_files()
print(f"{len(baseline)} baseline shards in the current release")
print(f"first 3: {baseline[:3]}")
print(f"last:    {baseline[-1]}")

1334 baseline shards in the current release
first 3: ['pubmed26n0001.xml.gz', 'pubmed26n0002.xml.gz', 'pubmed26n0003.xml.gz']
last:    pubmed26n1334.xml.gz


Daily update files extend the baseline:

In [7]:
updates = pubmed.list_update_files()
print(f"{len(updates)} update shards waiting to be applied on top of the baseline")
if updates:
    print(f"first 3: {updates[:3]}")

114 update shards waiting to be applied on top of the baseline
first 3: ['pubmed26n1335.xml.gz', 'pubmed26n1336.xml.gz', 'pubmed26n1337.xml.gz']


`download_baseline_file` (and its `download_update_file` sibling)
streams one shard to `~/.cache/biodb/pubmed/baseline/`, optionally
verifying the companion `.md5` checksum. Not executed here — each
shard is ~20 MB.

```python
path = pubmed.download_baseline_file(baseline[0])
# → ~/.cache/biodb/pubmed/baseline/pubmed26n0001.xml.gz
# `verify_md5=True` (default) fetches the .md5 sibling and checks the digest.
```

## 3. The XML parser — works on real shards and on `efetch` responses

`parse_pubmed_xml(path)` uses `xml.etree.iterparse`, so memory stays
roughly flat even on the multi-GB shards. Each row is one
`<PubmedArticle>`, with the same field shape as `query_abstract`.

We demonstrate against a tiny synthetic XML rather than a 20 MB
shard — the parser logic is identical.

In [8]:
import tempfile
from pathlib import Path

tiny_xml = b"""<?xml version="1.0"?>
<PubmedArticleSet>
  <PubmedArticle>
    <MedlineCitation>
      <PMID>1</PMID>
      <Article>
        <Journal><ISOAbbreviation>Nature</ISOAbbreviation>
          <JournalIssue><PubDate><Year>2026</Year></PubDate></JournalIssue></Journal>
        <ArticleTitle>Toy article one.</ArticleTitle>
        <Abstract><AbstractText>Short abstract.</AbstractText></Abstract>
        <AuthorList><Author><LastName>Curie</LastName><Initials>M</Initials></Author></AuthorList>
      </Article>
      <MeshHeadingList><MeshHeading><DescriptorName>Humans</DescriptorName></MeshHeading></MeshHeadingList>
    </MedlineCitation>
  </PubmedArticle>
  <PubmedArticle>
    <MedlineCitation>
      <PMID>2</PMID>
      <Article>
        <Journal><ISOAbbreviation>Science</ISOAbbreviation>
          <JournalIssue><PubDate><Year>2026</Year></PubDate></JournalIssue></Journal>
        <ArticleTitle>Toy article two.</ArticleTitle>
      </Article>
    </MedlineCitation>
  </PubmedArticle>
</PubmedArticleSet>
"""
tmp = Path(tempfile.mkdtemp()) / "tiny.xml"
tmp.write_bytes(tiny_xml)

df = pubmed.parse_pubmed_xml(tmp)
df[["pmid", "title", "journal", "pub_year", "authors", "mesh_terms"]]

,pmid,title,journal,pub_year,authors,mesh_terms
0,1,Toy article one.,Nature,2026,[Curie M],[Humans]
1,2,Toy article two.,Science,2026,[],[]


In production, point the parser at a real shard:

```python
articles = pubmed.parse_pubmed_xml(path)  # ~30 k articles per shard
```

For very large jobs, iterate shards rather than building one giant
DataFrame — the iterative `xml.etree.iterparse` parser handles
millions of articles cleanly.